# 02 - ACE Training
Train the localized ACE model on the dataset and log metrics.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

torch.manual_seed(42)

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.ace_wrapper import ACEWrapper
from src.trainer import BenchmarkTrainer


In [2]:
# Load Data
train_ds = MDTrajectoryDataset("../data/train.extxyz", cutoff=5.0)
val_ds = MDTrajectoryDataset("../data/val.extxyz", cutoff=5.0)
test_ds = MDTrajectoryDataset("../data/test.extxyz", cutoff=5.0)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)


Pre-computing graphs for 1000 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).


In [3]:
# Initialize ACE model (shared training setup, model-specific architecture)
model = ACEWrapper(
    num_elements=120,
    num_radial=8,
    l_max=2,
    r_cut=5.0,
    hidden_dim=32
)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = BenchmarkTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda" if torch.cuda.is_available() else "cpu",
    energy_weight=1.0,
    force_weight=100.0
)


C:\Users\Prabhat\AppData\Local\Programs\Python\Python314\Lib\ast.py:506: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
C:\Users\Prabhat\AppData\Local\Programs\Python\Python314\Lib\ast.py:506: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)


In [4]:
# Train + held-out test evaluation
metrics_df = trainer.train(max_epochs=50, patience=10)
metrics_df.to_csv("../data/ace_metrics.csv", index=False)

test_metrics = trainer.test_epoch(test_loader)
ace_test_df = pd.DataFrame([test_metrics])
ace_test_df.to_csv("../data/ace_test_metrics.csv", index=False)

metrics_df.head()

Epoch 000 | Time: 3.08s | Train E MAE: 16.52 meV/atom | Train F MAE: 207.58 meV/Å | Val E MAE: 15.35 meV/atom | Val F MAE: 165.69 meV/Å
Epoch 001 | Time: 1.79s | Train E MAE: 12.16 meV/atom | Train F MAE: 100.41 meV/Å | Val E MAE: 5.08 meV/atom | Val F MAE: 77.39 meV/Å
Epoch 002 | Time: 1.81s | Train E MAE: 4.79 meV/atom | Train F MAE: 70.14 meV/Å | Val E MAE: 1.82 meV/atom | Val F MAE: 66.99 meV/Å
Epoch 003 | Time: 1.71s | Train E MAE: 1.77 meV/atom | Train F MAE: 62.23 meV/Å | Val E MAE: 1.35 meV/atom | Val F MAE: 58.41 meV/Å
Epoch 004 | Time: 1.78s | Train E MAE: 7.89 meV/atom | Train F MAE: 52.55 meV/Å | Val E MAE: 10.40 meV/atom | Val F MAE: 47.42 meV/Å
Epoch 005 | Time: 1.75s | Train E MAE: 6.48 meV/atom | Train F MAE: 42.20 meV/Å | Val E MAE: 3.91 meV/atom | Val F MAE: 37.10 meV/Å
Epoch 006 | Time: 1.67s | Train E MAE: 5.92 meV/atom | Train F MAE: 31.03 meV/Å | Val E MAE: 38.84 meV/atom | Val F MAE: 25.05 meV/Å
Epoch 007 | Time: 1.73s | Train E MAE: 12.24 meV/atom | Train F MAE:

,epoch,loss,e_mae,f_mae,time,val_loss,val_e_mae,val_f_mae
0,0,6.687318,16.520141,207.580878,3.081537,4.160478,15.354179,165.692336
1,1,1.682459,12.157816,100.413747,1.785459,0.934843,5.084173,77.393116
2,2,0.758671,4.794824,70.141893,1.813614,0.687394,1.821769,66.987329
3,3,0.597507,1.769902,62.232529,1.714005,0.526126,1.351776,58.407654
4,4,0.437074,7.890208,52.550265,1.777784,0.355343,10.404509,47.424548
